# Step 1 — Baseline Network Analysis

Purpose: establish the original CIGRE MV network state and identify critical elements under the Step-1 N-1 switching procedure.

### What this cell does — Imports

Loads the original network builder and contingency routine. Step 1 intentionally does **not** use the all-closed Step-2 topology.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandapower as pp
from src import config
from src.network.model import build_network
from src.network.contingency import run_n1


### What this cell does — Baseline AC power flow

Creates the original CIGRE MV benchmark, solves AC power flow and extracts voltage, line loading and transformer loading. This is the reference state before depot profiles are added.

In [ ]:
net = build_network(halve_existing_loads=False, close_ring_switches=False)
pp.runpp(net)

bus_results = net.res_bus[["vm_pu", "va_degree"]].copy()
line_results = net.res_line[["p_from_mw", "q_from_mvar", "p_to_mw", "q_to_mvar", "loading_percent", "pl_mw"]].copy()
trafo_results = net.res_trafo[["loading_percent", "pl_mw"]].copy() if len(net.res_trafo) else None

print("Minimum voltage [pu]:", bus_results.vm_pu.min())
print("Maximum line loading [%]:", line_results.loading_percent.max())
if trafo_results is not None:
    print("Maximum transformer loading [%]:", trafo_results.loading_percent.max())
display(bus_results)


### What this cell does — Step-1 N-1

For each of the 12 ordinary lines, the outage is temporary. The routine evaluates the four Step-1 switching configurations, records voltage/loading/islanding, restores the line, and moves to the next case. The corrected tie-line names are excluded from the 12 ordinary outages.

In [ ]:
n1_summary, n1_violations = run_n1(net)
print("Cases:", len(n1_summary))
print("Electrical limits OK:", int(n1_summary.electrical_limits_ok.sum()), "/", len(n1_summary))
print("Full supply OK:", int(n1_summary.full_supply_ok.sum()), "/", len(n1_summary))
display(n1_summary)


### What this cell does — Export

Writes the reproducible Step-1 evidence used in the report.

In [ ]:
out = config.RESULTS / "step1_baseline"
out.mkdir(parents=True, exist_ok=True)
bus_results.to_csv(out / "bus_voltages.csv")
line_results.to_csv(out / "line_results.csv")
if trafo_results is not None: trafo_results.to_csv(out / "transformer_results.csv")
n1_summary.to_csv(out / "n1_summary.csv", index=False)
n1_violations.to_csv(out / "n1_violations.csv", index=False)
print("Saved to", out)
